# Latent-Space Concatenation (~20s)

Generate multiple sound variations and **concatenate them in latent space**
using the LSD model itself — not just audio-level crossfading.

### Approach
1. Train the pipeline (1s clips, same as notebook 03)
2. Generate 3 sets of raw audio segments (different seeds/BPS configs)
3. **Encode** each variation back to EnCodec latent space
4. **Concatenate the latents** with crossfade overlap in latent space
5. **Decode the long latent** through the graph decoder in a single pass
6. Apply BPS modulation + pedalboard effects on the full ~20s output

### Why latent-space concatenation?
Audio-level crossfading blends waveforms, which can produce phase artifacts
and discontinuities. Latent-space concatenation lets the graph decoder's
project→gate→lift mechanism smooth the transition across the boundary,
because the decoder sees the full sequence at once and the
`WaveReconstructionBlock` pools over the entire temporal axis.

### Knobs
All knobs from notebook 03, plus:
- `NUM_VARIATIONS` — number of segments to concatenate (3 → ~18-20s)
- `LATENT_OVERLAP_FRAC` — crossfade overlap in latent space (0.1-0.5)
- Per-variation BPS/effect settings defined in `VARIATIONS` list

In [ ]:
# --- Generation knobs ---
SEED = 42
STEPS = 50
TEMPERATURE = 1.0
USE_C_SPEC = True

# --- Concatenation knobs ---
NUM_VARIATIONS = 3         # number of variations to concatenate
LATENT_OVERLAP_FRAC = 0.25 # crossfade overlap in latent space
CLIP_SECONDS = 1.0         # training clip length
SEGMENT_SECONDS = 6        # per-variation target length
OVERLAP_FRAC = 0.5         # audio overlap-and-add within each variation
SAMPLE_RATE = 24000

# --- BPS modulation (shared across all variations) ---
MOD_WAVEFORM = 'sine'

# --- Per-variation settings ---
# Each entry: (label, seed, bps, trem_depth, vib_depth, sweep_min, sweep_max,
#             reverb_room, delay_sec, dist_drive)
VARIATIONS = [
    ('Ambient',   200, 1.5, 0.2, 3.0, 200,  2000, 0.7, 0.5,   0),
    ('Gritty',    300, 4.0, 0.5, 0.5, 500,  5000, 0.3, 0.1875, 15),
    ('Dreamy',     42, 2.0, 0.4, 2.0, 300,  8000, 0.5, 0.375,  0),
]

# --- Training knobs ---
DECODER_EPOCHS = 20
DIFFUSION_EPOCHS = 20
NUM_SAMPLES = 64

CLIP_LENGTH = int(CLIP_SECONDS * SAMPLE_RATE)
TOTAL_LENGTH = int(SEGMENT_SECONDS * SAMPLE_RATE)
STRIDE = 320
LATENT_LENGTH_PER_CLIP = CLIP_LENGTH // STRIDE
print(f'Target: {NUM_VARIATIONS} x {SEGMENT_SECONDS}s = ~{NUM_VARIATIONS * SEGMENT_SECONDS}s')
print(f'Latent overlap: {LATENT_OVERLAP_FRAC} ({int(LATENT_LENGTH_PER_CLIP * SEGMENT_SECONDS * LATENT_OVERLAP_FRAC)} frames)')
print(f'Variations: {[v[0] for v in VARIATIONS]}')

In [ ]:
import math
import numpy as np
import scipy.signal as scisig
import torch
import torch.nn as nn
import torchaudio
from IPython.display import Audio, display
import matplotlib.pyplot as plt
from pedalboard import (
    Pedalboard, Compressor, Reverb, Delay, Chorus,
    Distortion, Phaser, Bitcrush, Gain, Limiter,
    LowShelfFilter, HighShelfFilter, PeakFilter,
)

from ald_sc.build_prior import build_arrow_prior
from ald_sc.audio_codec import BaselineAudioDecoder, AudioVAE
from ald_sc.graph_decoder import GraphDecoder
from ald_sc.dit import MinimalDiT
from ald_sc.data import ToyAudioDataset, build_audio_dataloader
from ald_sc.losses import ALDSCLoss
from ald_sc.schedule import CosineSchedule
from ald_sc.sampling import sample_ddim
from ald_sc.trainer import train_audio_decoder, train_audio_diffusion

device = torch.device('cpu')
torch.manual_seed(SEED)
print('Ready. Device:', device)

## Step 1: Train the pipeline (1s clips)

In [ ]:
class StubEncoder(nn.Module):
    def __init__(self, latent_dim=128, stride=320):
        super().__init__()
        self.proj = nn.Conv1d(1, latent_dim, stride, stride=stride)
    def encode(self, x, prior):
        z = self.proj(x).float()
        a = z.mean(dim=2)
        c_spec = prior.chart_energy_descriptor(a)
        return z, a, c_spec
    def extract_features(self, x):
        return self.proj(x).float()

encoder = StubEncoder()
latent_length = CLIP_LENGTH // STRIDE

dataset = ToyAudioDataset(num_samples=NUM_SAMPLES, audio_length=CLIP_LENGTH)
loader = build_audio_dataloader(dataset, batch_size=8, shuffle=False)
features = [encoder.extract_features(b).mean(dim=2) for b in loader]
embeddings = torch.cat(features, dim=0)
prior = build_arrow_prior(embeddings, q=8, k=4)
print(f'Prior: q={prior.q}')

graph_decoder = GraphDecoder(
    latent_channels=128, out_channels=1, feature_dim=128,
    base_channels=32, prior=prior, upsample_strides=(2, 4, 5, 8),
)
loss_fn = ALDSCLoss(prior=prior, lambda_rec=1.0, lambda_stft=0.0,
                    lambda_chart=0.5, lambda_smooth=0.1)
train_loader = build_audio_dataloader(dataset, batch_size=4, shuffle=True)

graph_vae = AudioVAE(encoder=encoder, decoder=graph_decoder)
print('Training graph decoder...')
gl = list(train_audio_decoder(train_loader, graph_vae, prior, loss_fn,
                              epochs=DECODER_EPOCHS, lr=1e-3, device=device))
print(f'  {gl[0]["loss"]:.4f} -> {gl[-1]["loss"]:.4f}')

dit = MinimalDiT(latent_channels=128, latent_length=latent_length,
                 patch_size=8, dim=64, depth=2, num_heads=4, spec_dim=24)
sched = CosineSchedule(num_steps=1000)
for p in graph_vae.parameters():
    p.requires_grad_(False)
print('Training DiT...')
dl = list(train_audio_diffusion(train_loader, graph_vae, dit, prior, sched,
                                epochs=DIFFUSION_EPOCHS, lr=1e-3, device=device))
print(f'  {dl[0]["loss"]:.4f} -> {dl[-1]["loss"]:.4f}')

## Step 2: Generate raw audio for each variation

For each variation, generate 1s segments and stitch with overlap-and-add
into a ~6s raw audio clip (before effects).

In [ ]:
def generate_segment(seed_val):
    z = sample_ddim(dit, sched, batch_size=1, steps=STEPS, seed=seed_val, device=device)
    z = z * TEMPERATURE
    a = z.mean(dim=2)
    c_spec = prior.chart_energy_descriptor(a)
    with torch.no_grad():
        audio = graph_decoder(z, c_spec if USE_C_SPEC else torch.zeros_like(c_spec))
    return audio.squeeze(0).squeeze(0)

def overlap_add(segments, overlap_frac):
    seg_len = segments[0].shape[-1]
    overlap = int(seg_len * overlap_frac)
    hop = seg_len - overlap
    total_len = hop * (len(segments) - 1) + seg_len
    output = torch.zeros(total_len)
    window = torch.linspace(0, 1, overlap)
    for i, seg in enumerate(segments):
        start = i * hop
        if i > 0:
            prev_start = start - overlap
            output[prev_start:prev_start + overlap] *= (1 - window)
        output[start:start + seg_len] += seg
        if i > 0:
            output[start:start + overlap] *= window
    return output

clip_samples = CLIP_LENGTH
hop_samples = int(clip_samples * (1 - OVERLAP_FRAC))
num_segments = max(2, math.ceil((TOTAL_LENGTH - clip_samples) / hop_samples) + 1)

# Generate raw audio for each variation
raw_audios = []
for i, (label, seed, *_) in enumerate(VARIATIONS):
    segs = [generate_segment(seed + j * 1000) for j in range(num_segments)]
    audio = overlap_add(segs, OVERLAP_FRAC)
    if audio.shape[-1] > TOTAL_LENGTH:
        audio = audio[:TOTAL_LENGTH]
    elif audio.shape[-1] < TOTAL_LENGTH:
        audio = torch.nn.functional.pad(audio, (0, TOTAL_LENGTH - audio.shape[-1]))
    audio = audio - audio.mean()  # DC-block: decoder emits large DC (PR #59)
    peak = audio.abs().max()
    if peak > 0:
        audio = audio / peak
    raw_audios.append(audio)
    print(f'  {label}: {audio.shape[-1]} samples ({audio.shape[-1]/SAMPLE_RATE:.1f}s)')

print(f'\nGenerated {len(raw_audios)} variations, each ~{SEGMENT_SECONDS}s')
print(f'Total if concatenated: ~{len(raw_audios) * SEGMENT_SECONDS}s')

In [ ]:
# Listen to each raw variation
for i, (label, *_) in enumerate(VARIATIONS):
    print(f'Variation {i+1}: {label}')
    display(Audio(raw_audios[i].numpy(), rate=SAMPLE_RATE))

## Step 3: Latent-space concatenation

This is the key step. Instead of crossfading audio waveforms, we:
1. **Encode** each variation back to latent space via the stub encoder
2. **Crossfade the latents** at their boundaries in latent space
3. **Decode the concatenated latent** through the graph decoder in one pass

The graph decoder sees the full ~18s latent at once. Its
`WaveReconstructionBlock` pools over the entire temporal axis, so the
project→gate→lift mechanism naturally smooths transitions across
variation boundaries.

In [ ]:
def encode_audio(encoder, audio, prior):
    """Encode audio waveform to latent z (B, C, T_latent)."""
    x = audio.unsqueeze(0).unsqueeze(0)  # (1, 1, T)
    with torch.no_grad():
        z = encoder.extract_features(x)  # (1, 128, T_latent)
    return z

def latent_crossfade(z_list, overlap_frac):
    """Concatenate latents with crossfade overlap in latent space.
    
    z_list: list of (1, C, T) tensors
    Returns: (1, C, T_total) concatenated tensor
    """
    if len(z_list) == 1:
        return z_list[0]
    
    T = z_list[0].shape[-1]
    overlap = max(1, int(T * overlap_frac))
    hop = T - overlap
    total_len = hop * (len(z_list) - 1) + T
    
    output = torch.zeros(1, z_list[0].shape[1], total_len)
    window = torch.linspace(0, 1, overlap)
    
    for i, z in enumerate(z_list):
        start = i * hop
        if i > 0:
            prev_start = start - overlap
            output[:, :, prev_start:prev_start + overlap] *= (1 - window)
        output[:, :, start:start + T] += z
        if i > 0:
            output[:, :, start:start + overlap] *= window
    
    return output

# Encode each variation to latent space
print('Encoding variations to latent space...')
z_list = []
for i, (label, *_) in enumerate(VARIATIONS):
    z = encode_audio(encoder, raw_audios[i], prior)
    z_list.append(z)
    print(f'  {label}: z={z.shape} ({z.shape[-1]} latent frames)')

# Concatenate in latent space with crossfade
z_concat = latent_crossfade(z_list, LATENT_OVERLAP_FRAC)
print(f'\nConcatenated latent: {z_concat.shape} ({z_concat.shape[-1]} frames)')
print(f'  = {z_concat.shape[-1] * STRIDE / SAMPLE_RATE:.1f}s of audio')

# Derive c_spec from the full concatenated latent
a_full = z_concat.mean(dim=2)
c_spec_full = prior.chart_energy_descriptor(a_full)
print(f'c_spec (from full latent): {c_spec_full.shape}')

# Decode the concatenated latent in one pass through the graph decoder
print('\nDecoding concatenated latent through graph decoder...')
with torch.no_grad():
    concat_audio = graph_decoder(z_concat, c_spec_full)
concat_audio = concat_audio.squeeze(0).squeeze(0)  # (T,)

concat_audio = concat_audio - concat_audio.mean()  # DC-block: decoder emits large DC (PR #59)
peak = concat_audio.abs().max()
if peak > 0:
    concat_audio = concat_audio / peak

print(f'Concatenated audio: {concat_audio.shape[-1]} samples '
      f'({concat_audio.shape[-1]/SAMPLE_RATE:.1f}s)')

In [ ]:
# Compare: latent-space concatenation vs naive audio concatenation
# Naive: just crossfade the raw audio at the same overlap
naive_audio = overlap_add(raw_audios, LATENT_OVERLAP_FRAC)
naive_audio = naive_audio - naive_audio.mean()  # DC-block: decoder emits large DC (PR #59)
peak = naive_audio.abs().max()
if peak > 0:
    naive_audio = naive_audio / peak

print(f'Latent-space concat: {concat_audio.shape[-1]/SAMPLE_RATE:.1f}s')
print(f'Naive audio concat:  {naive_audio.shape[-1]/SAMPLE_RATE:.1f}s')

fig, axes = plt.subplots(2, 1, figsize=(14, 5))
t1 = torch.arange(len(concat_audio)) / SAMPLE_RATE
axes[0].plot(t1.numpy(), concat_audio.numpy(), alpha=0.7, linewidth=0.3)
axes[0].set_title('Latent-space concatenation (graph decoder sees full sequence)')
axes[0].set_ylabel('Amplitude')

t2 = torch.arange(len(naive_audio)) / SAMPLE_RATE
axes[1].plot(t2.numpy(), naive_audio.numpy(), alpha=0.7, linewidth=0.3, color='orange')
axes[1].set_title('Naive audio crossfade')
axes[1].set_ylabel('Amplitude')
axes[1].set_xlabel('Time (s)')

# Mark variation boundaries
for ax in axes:
    for i in range(1, NUM_VARIATIONS):
        boundary = i * SEGMENT_SECONDS
        ax.axvline(x=boundary, color='red', linestyle='--', alpha=0.3, label=f'V{i+1} boundary' if i == 1 else '')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
print('Latent-space concatenation:')
display(Audio(concat_audio.numpy(), rate=SAMPLE_RATE))
print('\nNaive audio crossfade:')
display(Audio(naive_audio.numpy(), rate=SAMPLE_RATE))

## Step 4: Apply per-section BPS modulation + effects

Now apply the BPS modulation and pedalboard effects from notebook 03,
but with **per-section settings** that change at each variation boundary.
The modulation parameters transition smoothly across boundaries.

In [ ]:
def make_lfo(num_samples, bps, sr, waveform='sine'):
    t = torch.arange(num_samples, dtype=torch.float32) / sr
    phase = 2 * math.pi * bps * t
    if waveform == 'sine':
        return torch.sin(phase)
    elif waveform == 'triangle':
        return 2 * (phase / (2 * math.pi) % 1) - 1
    elif waveform == 'square':
        return torch.sign(torch.sin(phase))
    elif waveform == 'sawtooth':
        return 2 * (phase / (2 * math.pi) % 1) - 1
    raise ValueError(f'Unknown waveform: {waveform}')

def apply_tremolo(audio, lfo, depth):
    gain = 1.0 - depth * (1.0 - lfo) / 2.0
    return audio * gain

def apply_vibrato(audio, lfo, depth_semitones, sr=24000, window_ms=10):
    if depth_semitones == 0:
        return audio
    window_samples = max(1, int(sr * window_ms / 1000))
    num_windows = max(1, len(audio) // window_samples)
    pitch_shifts = lfo[:num_windows] * depth_semitones
    ratios = 2.0 ** (pitch_shifts / 12.0)
    chunks = []
    for i in range(num_windows):
        start = i * window_samples
        end = min(start + window_samples, len(audio))
        chunk = audio[start:end]
        if len(chunk) == 0:
            break
        new_len = max(1, int(len(chunk) / ratios[i].item()))
        r = torch.nn.functional.interpolate(
            chunk.unsqueeze(0).unsqueeze(0), size=new_len, mode='linear', align_corners=False
        )
        chunks.append(r.squeeze())
    return torch.cat(chunks)

def make_sectional_lfo(total_samples, sr, section_lengths_s, bps_list, waveform='sine'):
    """Build an LFO that changes BPS rate at each section boundary.
    Uses phase-continuous stitching to avoid clicks."""
    lfo = torch.zeros(total_samples)
    phase_acc = 0.0
    sample_pos = 0
    for sec_len, bps in zip(section_lengths_s, bps_list):
        n = int(sec_len * sr)
        n = min(n, total_samples - sample_pos)
        if n <= 0:
            break
        phase_inc = 2 * math.pi * bps / sr
        for j in range(n):
            phase = phase_acc + j * phase_inc
            if waveform == 'sine':
                lfo[sample_pos + j] = math.sin(phase)
            elif waveform == 'triangle':
                p = (phase / (2 * math.pi)) % 1
                lfo[sample_pos + j] = 2 * p - 1
            elif waveform == 'square':
                lfo[sample_pos + j] = 1.0 if math.sin(phase) >= 0 else -1.0
            elif waveform == 'sawtooth':
                p = (phase / (2 * math.pi)) % 1
                lfo[sample_pos + j] = 2 * p - 1
        phase_acc = phase_acc + n * phase_inc
        sample_pos += n
    return lfo

# Build sectional LFO with per-variation BPS
bps_list = [v[2] for v in VARIATIONS]  # BPS from each variation
section_lengths = [SEGMENT_SECONDS] * NUM_VARIATIONS

# Use the latent-concatenated audio length
total_samples = len(concat_audio)
lfo = make_sectional_lfo(total_samples, SAMPLE_RATE, section_lengths, bps_list, MOD_WAVEFORM)

print(f'Sectional LFO: BPS changes at boundaries = {bps_list}')
print(f'Total LFO length: {total_samples} samples ({total_samples/SAMPLE_RATE:.1f}s)')

# Plot LFO
fig, ax = plt.subplots(figsize=(14, 2))
t = torch.arange(total_samples) / SAMPLE_RATE
ax.plot(t.numpy(), lfo.numpy(), linewidth=0.3)
for i in range(1, NUM_VARIATIONS):
    ax.axvline(x=i * SEGMENT_SECONDS, color='red', linestyle='--', alpha=0.3)
ax.set_title(f'Sectional LFO (BPS: {bps_list})')
ax.set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

In [ ]:
# Apply tremolo and vibrato with per-section depths
# We need per-sample depth interpolation across sections
def make_sectional_param(total_samples, sr, section_lengths_s, values):
    """Build a parameter curve that steps at each section boundary."""
    curve = torch.zeros(total_samples)
    pos = 0
    for sec_len, val in zip(section_lengths_s, values):
        n = min(int(sec_len * sr), total_samples - pos)
        curve[pos:pos + n] = val
        pos += n
    # Fill remainder
    if pos < total_samples:
        curve[pos:] = values[-1]
    return curve

# Per-section depth parameters
trem_depths = [v[3] for v in VARIATIONS]
vib_depths = [v[4] for v in VARIATIONS]

# Apply tremolo with per-section depth
trem_curve = make_sectional_param(total_samples, SAMPLE_RATE, section_lengths, trem_depths)
trem_gain = 1.0 - trem_curve * (1.0 - lfo) / 2.0
audio_mod = concat_audio * trem_gain

# Apply vibrato (use max depth for windowed resampling, per-section would require
# more complex windowed approach — we use the overall LFO with max depth)
max_vib = max(vib_depths)
if max_vib > 0:
    audio_mod = apply_vibrato(audio_mod, lfo, max_vib, SAMPLE_RATE)

audio_mod = audio_mod - audio_mod.mean()  # DC-block: decoder emits large DC (PR #59)
peak = audio_mod.abs().max()
if peak > 0:
    audio_mod = audio_mod / peak

print(f'After BPS modulation (sectional): {audio_mod.shape[-1]/SAMPLE_RATE:.1f}s')
display(Audio(audio_mod.numpy(), rate=SAMPLE_RATE))

In [ ]:
# Apply pedalboard effects with per-section settings
# We process each section separately through its own pedalboard chain,
# then crossfade at boundaries

def build_board(variation_params):
    """Build pedalboard from variation tuple."""
    (_, _, _, _, _, _, _, reverb_room, delay_sec, dist_drive) = variation_params
    effects = [Compressor(threshold_db=-20, ratio=4)]
    if dist_drive > 0:
        effects.append(Distortion(drive_db=dist_drive))
    effects.append(Delay(delay_seconds=delay_sec, mix=0.25))
    effects.append(Reverb(room_size=reverb_room))
    effects.append(Limiter(threshold_db=-1.0))
    return Pedalboard(effects)

# Process each section through its own pedalboard
section_audio = []
fade_samples = int(SAMPLE_RATE * 0.1)  # 100ms crossfade between sections
window = torch.linspace(0, 1, fade_samples)

pos = 0
for i, var in enumerate(VARIATIONS):
    sec_samples = min(int(SEGMENT_SECONDS * SAMPLE_RATE), len(audio_mod) - pos)
    chunk = audio_mod[pos:pos + sec_samples].numpy().astype(np.float32)
    
    board = build_board(var)
    processed = torch.from_numpy(board(chunk, SAMPLE_RATE))
    section_audio.append(processed)
    pos += sec_samples

# Crossfade sections
final_audio = section_audio[0]
for i in range(1, len(section_audio)):
    # Overlap crossfade
    prev_tail = final_audio[-fade_samples:]
    curr_head = section_audio[i][:fade_samples]
    blended = prev_tail * (1 - window) + curr_head * window
    final_audio = torch.cat([
        final_audio[:-fade_samples],
        blended,
        section_audio[i][fade_samples:]
    ])

final_audio = final_audio - final_audio.mean()  # DC-block: decoder emits large DC (PR #59)
peak = final_audio.abs().max()
if peak > 0:
    final_audio = final_audio / peak

print(f'Final audio: {final_audio.shape[-1]} samples ({final_audio.shape[-1]/SAMPLE_RATE:.1f}s)')
display(Audio(final_audio.numpy(), rate=SAMPLE_RATE))

## Step 5: Full pipeline visualization

In [ ]:
stages = [
    ('1. Raw variations (stacked)', None),  # special handling
    ('2. Latent-space concatenation', concat_audio),
    ('3. After BPS modulation', audio_mod),
    ('4. After pedalboard (per-section)', final_audio),
]

fig, axes = plt.subplots(len(stages), 1, figsize=(14, 2.5 * len(stages)))

# Stage 1: stacked raw variations
ax = axes[0]
offset = 0
colors = ['blue', 'green', 'orange']
for i, (audio, var) in enumerate(zip(raw_audios, VARIATIONS)):
    t = (torch.arange(len(audio)) + offset) / SAMPLE_RATE
    ax.plot(t.numpy(), audio.numpy() + (len(raw_audios) - i) * 1.5,
            alpha=0.6, linewidth=0.3, color=colors[i % len(colors)],
            label=f'{var[0]} (seed={var[1]})')
    offset += len(audio)
ax.set_title('1. Raw variations (each ~6s)')
ax.legend(fontsize=7, loc='upper right')
ax.set_ylabel('Amplitude (offset)')

# Remaining stages
for ax, (name, audio) in zip(axes[1:], stages[1:]):
    t = torch.arange(len(audio)) / SAMPLE_RATE
    ax.plot(t.numpy(), audio.numpy(), alpha=0.7, linewidth=0.3)
    ax.set_title(name)
    ax.set_ylabel('Amplitude')
    for i in range(1, NUM_VARIATIONS):
        ax.axvline(x=i * SEGMENT_SECONDS, color='red', linestyle='--', alpha=0.2)

axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

In [ ]:
# Export to WAV
import torchaudio
import soundfile as sf
from pathlib import Path

out_dir = Path('results')
out_dir.mkdir(exist_ok=True)

# Save final output
sf.write(str(out_dir / 'latent_concat_20s.wav'),
         final_audio.numpy(), SAMPLE_RATE)
print(f'Saved: {out_dir / "latent_concat_20s.wav"} ({final_audio.shape[-1]/SAMPLE_RATE:.1f}s)')

# Also save individual raw variations
for i, (label, *_) in enumerate(VARIATIONS):
    path = out_dir / f'variation_{i+1}_{label.lower()}.wav'
    sf.write(str(path), raw_audios[i].numpy(), SAMPLE_RATE)
    print(f'Saved: {path}')

## Summary

This notebook demonstrates **latent-space concatenation** — using the LSD
model itself to join multiple sound variations into a continuous ~20s piece:

1. **Generate** 3 variations (different seeds, ~6s each)
2. **Encode** each variation back to EnCodec latent space
3. **Crossfade latents** at boundaries in latent space
4. **Decode the full concatenated latent** through the graph decoder in one pass
5. **Apply per-section BPS modulation** (tremolo, vibrato) with phase-continuous LFO
6. **Apply per-section pedalboard effects** with crossfade between sections

### Why latent-space concatenation works

The graph decoder's `WaveReconstructionBlock` pools over the **entire** temporal
axis when computing the feature field `A`. When the decoder sees a 18s latent
instead of a 1s latent, the project→gate→lift mechanism uses the full spectral
context — the `c_spec` derived from all 3 variations. This naturally smooths
transitions at boundaries, because the gating weights are computed from the
global energy distribution, not local frame-by-frame.

### Knobs

| Knob | What it controls |
|------|-------------------|
| `NUM_VARIATIONS` | Number of segments to concatenate (3 → ~18-20s) |
| `LATENT_OVERLAP_FRAC` | Crossfade overlap in latent space (0.1-0.5) |
| `VARIATIONS` | Per-section: seed, BPS, tremolo/vibrato depth, FX params |
| `MOD_WAVEFORM` | LFO shape shared across sections (phase-continuous) |

### Next steps
- Train DiT on longer latents directly (avoid overlap-and-add within sections)
- Add CLAP conditioning per section (Phase 2)
- Use real EnCodec for richer latent space
- Integrate with the DSL composition system (issue #18)